# Task 2 — Fine-tuning del solo mask decoder (Kaggle, GPU)

Baseline del progetto: si addestra solo il **mask decoder** (~4M parametri); image encoder e prompt encoder restano congelati.
L'encoder congelato dà sempre lo stesso embedding, quindi lo calcoliamo **una volta** e il training usa solo il decoder.

Riferimenti da battere (Task 1, test, box esatti): **0.920** uscita singola · **0.937** 3 maschere.

**Session options:** *Accelerator* **GPU T4 x2** (ne usiamo una) · *Internet* **On** · *Persistence* **Files only**.

Durata stimata: ~30 minuti (cache ~8 min, training da 30 epoche, valutazioni). Per non tenere aperta la scheda: *Save Version → Save & Run All*.

## 1. Codice della repo

In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/caMatt99/sam-peft-kvasir.git"
WORK = Path("/kaggle/working")
REPO = WORK / "sam-peft-kvasir"

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}
!pip install -q -e "{REPO}[dev]"
%cd {REPO}
sys.path.insert(0, str(REPO / "src"))   # il kernel già avviato non vede l'install editable
!git log --oneline -1

## 2. Dati e pesi

In [ ]:
import torch
assert torch.cuda.is_available(), "Accendi la GPU in Session options → Accelerator"
print(torch.cuda.get_device_name(0))

def find_kvasir_root():
    for base in [Path("/kaggle/input"), WORK / "data"]:
        if base.exists():
            for p in base.rglob("images"):
                if (p.parent / "masks").is_dir():
                    return p.parent
    return None

RAW = find_kvasir_root()
if RAW is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q -L --no-check-certificate https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/ && rm /kaggle/working/data/kvasir-seg.zip
    RAW = find_kvasir_root()
assert RAW is not None

PROCESSED = WORK / "processed"
if not (PROCESSED / "meta.json").exists():
    !python scripts/prepare_data.py --raw-root {RAW} --out {PROCESSED} > /dev/null

CKPT = WORK / "checkpoints" / "sam_vit_b_01ec64.pth"
if not CKPT.exists() or CKPT.stat().st_size < 300e6:
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    !wget -q -L https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {CKPT}

CACHE = WORK / "cache" / "vit_b"
RUN = "runs/decoder"
print(RAW, "|", PROCESSED, "| checkpoint MB:", round(CKPT.stat().st_size / 1e6))

## 3. Test
Nuovi in questa task (`tests/test_training.py`): la validation con la cache dà le stesse metriche dell'encoder, il metodo `decoder` congela tutto tranne il decoder, i pesi salvati si ricaricano identici, lo script di training gira da capo a fondo.

In [ ]:
!python -m pytest -q

## 4. Cache degli embedding
Encoder originale in float32 su tutte le 1000 immagini, salvato in float16 (~2 GB). Si fa una volta: se la cartella esiste già viene saltato.
Il tempo per immagine è il costo che i metodi PEFT (Task 3–5) pagheranno **a ogni epoca**.

In [ ]:
!python scripts/cache_embeddings.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --out {CACHE}
!du -sh {CACHE}

## 5. Zero-shot sul test (di nuovo)
Gli stessi numeri della Task 1, rifatti qui per avere i risultati **per immagine** accanto a quelli del fine-tuning: così il confronto si fa immagine per immagine (confronto appaiato), che è molto più preciso del confronto tra due medie.

In [ ]:
for flag, name in [("", "zeroshot"), ("--multimask", "zeroshot_multimask")]:
    !python scripts/evaluate.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --split test {flag} --out runs/{name}

## 6. Training
Un solo run (seed 0: fissa ordine delle immagini e jitter dei box, quindi il risultato è riproducibile).
**Controllo:** l'epoca 0 è la validation prima del training e deve dare ~0.906 (lo zero-shot della Task 1 su val). Se è molto diversa, la cache è sbagliata: fermati.
`*` = nuova migliore epoca sulla validation.

In [ ]:
!python scripts/train.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --cache {CACHE} --seed 0 --out {RUN}

## 7. Valutazione finale
Con lo stesso `evaluate.py` della Task 1: encoder vero in float32, box fissi. Il test si guarda **solo qui**, dopo aver scelto l'epoca sulla validation.

In [ ]:
for split_name in ["val", "test"]:
    !python scripts/evaluate.py --raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT} --split {split_name} --weights {RUN}/trainable.pt --out {RUN}

## 8. Analisi
### Curve di training

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

h = pd.DataFrame(json.loads(Path(f"{RUN}/history.json").read_text()))
summ = json.loads(Path(f"{RUN}/summary.json").read_text())
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].plot(h["epoch"][1:], h["train_loss"][1:])
ax[1].plot(h["epoch"], h["val_dice"]); ax[1].axhline(0.906, ls="--", c="gray", label="zero-shot")
ax[2].plot(h["epoch"], h["val_dice_jitter"]); ax[2].axhline(0.882, ls="--", c="gray", label="zero-shot")
for a, t in zip(ax, ["loss di training", "val Dice, box esatti", "val Dice, box con jitter"]):
    a.set_title(t); a.set_xlabel("epoca")
ax[1].legend(); ax[2].legend()
plt.show()

print(f"epoca 0 (controllo cache): {h['val_dice'][0]:.3f} → atteso ~0.906")
print(f"epoca migliore: {summ['best_epoch']} · val Dice {summ['best_val_dice']:.3f}")

### Risultati sul test
Dice medio ± errore standard, tra parentesi il tasso di fallimento (Dice < 0.5).

In [ ]:
def load(run, split_name="test"):
    return json.loads(Path(f"runs/{run}/{split_name}/metrics.json").read_text())

rows = []
for g in [g for g in ["all", "single_polyp", "multi_polyp", "large_polyp"] if g in load("zeroshot")["groups"]]:
    row = {"gruppo": g}
    for name, run in [("zero-shot", "zeroshot"), ("zero-shot 3 masch.", "zeroshot_multimask")]:
        m = load(run)["groups"][g]
        row[name] = f"{m['boxes.dice_raw']['mean']:.3f} ({m['boxes.fail_rate']:.0%})"
        row[name + " jitter"] = f"{m['boxes_jitter.dice_raw']['mean']:.3f}"
    m = load("decoder")["groups"][g]
    row["decoder"] = f"{m['boxes.dice_raw']['mean']:.3f} ± {m['boxes.dice_raw']['sem']:.3f} ({m['boxes.fail_rate']:.0%})"
    row["decoder jitter"] = f"{m['boxes_jitter.dice_raw']['mean']:.3f} ± {m['boxes_jitter.dice_raw']['sem']:.3f}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("gruppo"))

print(f"parametri addestrati: {summ['parameters']['trainable']:,} su {summ['parameters']['total']:,}")
print(f"training: {summ['train_s_per_epoch_median']:.1f} s per epoca (mediana) · "
      f"picco memoria {summ['peak_gpu_memory_mb']:.0f} MB")

### Confronto appaiato, immagine per immagine (test, box esatti)
Per ogni immagine: Dice del decoder meno Dice zero-shot. L'errore standard della **differenza** è molto più piccolo di quello delle due medie, perché la difficoltà delle singole immagini si cancella.

In [ ]:
per = lambda run: pd.read_csv(f"runs/{run}/test/per_image.csv").set_index("id")
zs, zm = per("zeroshot"), per("zeroshot_multimask")
dec = per("decoder").loc[zs.index]

for ref_name, ref in [("zero-shot", zs), ("zero-shot 3 maschere", zm)]:
    for k in ["boxes", "boxes_jitter"]:
        diff = dec[f"{k}.dice_raw"] - ref[f"{k}.dice_raw"]
        print(f"decoder − {ref_name:<21} {k:<13} {diff.mean():+.3f} ± {diff.std(ddof=1) / np.sqrt(len(diff)):.3f}"
              f"   migliora {(diff > 0.01).sum():3d} · peggiora {(diff < -0.01).sum():3d} · pari {(diff.abs() <= 0.01).sum():3d}")

### Per dimensione del polipo (test)

In [ ]:
tab = pd.DataFrame({"area": zs["area_frac"], "zero-shot": zs["boxes.dice_raw"],
                    "3 maschere": zm["boxes.dice_raw"], "decoder": dec["boxes.dice_raw"],
                    "decoder jitter": dec["boxes_jitter.dice_raw"]})
tab["bin"] = pd.cut(tab["area"], [0, .1, .25, .5, 1], labels=["<10%", "10-25%", "25-50%", ">50%"])
display(tab.groupby("bin", observed=True).agg(n=("area", "size"), **{c: (c, "mean") for c in
        ["zero-shot", "3 maschere", "decoder", "decoder jitter"]}).round(3))

### I fallimenti della Task 1, dopo il fine-tuning
Le 4 immagini dove SAM zero-shot falliva (Dice < 0.5, uscita singola). Predizione del decoder addestrato.

In [ ]:
import cv2
from matplotlib.patches import Rectangle
from sampeft.data.kvasir import KvasirDataset
from sampeft.model import decode, encode, load_finetuned, load_sam

split = json.loads(Path("data/splits/split.json").read_text())
FAILS = {"cju42py9mlqyd0818u3d1d7ga": "val", "cju16whaj0e7n0855q7b6cjkm": "val",
         "cju6wi3akvn8r0801px8eligc": "test", "cju88l66no10s0850rsda7ej1": "val"}
sam0 = load_sam(CKPT, device="cuda").eval()
sam1 = load_sam(CKPT, device="cuda").eval()
load_finetuned(sam1, f"{RUN}/trainable.pt")

for img_id, s in FAILS.items():
    ds = KvasirDataset(RAW, PROCESSED, [img_id], train=False, prompts_file=f"data/splits/prompts_{s}.json")
    x = ds[0]
    gt = x["gt"].numpy()
    preds = []
    with torch.no_grad():
        emb = encode(sam0, x["image"][None].cuda())
        for m in (sam0, sam1):
            logits, _ = decode(m, emb[0], x["boxes"].cuda(), x["input_size"], x["original_size"])
            preds.append((logits > 0).any(0).cpu().numpy())
    img = cv2.cvtColor(cv2.imread(str(RAW / "images" / f"{img_id}.jpg")), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    ax[0].imshow(img)
    for b in x["boxes_orig"].numpy():
        ax[0].add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="lime", lw=2))
    ax[0].set_title(f"{img_id}\n{s}", fontsize=9); ax[1].imshow(gt, cmap="gray"); ax[1].set_title("GT")
    for a, p, t in zip(ax[2:], preds, ["zero-shot", "decoder"]):
        d = 2 * (p & gt).sum() / (p.sum() + gt.sum())
        a.imshow(p, cmap="gray"); a.set_title(f"{t}  Dice {d:.3f}")
    for a in ax: a.axis("off")
    plt.show()

## 9. Salvataggio
Crea `/kaggle/working/task2_results.zip` (metriche, curve, pesi addestrati: ~16 MB, senza la cache).
Per scaricarlo: pannello a destra **Output** → `task2_results.zip` → menu ⋮ → **Download**. Non va su GitHub.

In [ ]:
!rm -f {WORK}/task2_results.zip   # zip aggiunge a un archivio esistente: si riparte da zero
!zip -qr {WORK}/task2_results.zip runs/
!ls -lh {WORK}/task2_results.zip

## Da annotare
1. L'epoca 0 ritrova lo zero-shot (~0.906 su val)? Se sì, la cache è corretta.
2. Dice sul test, box esatti e con jitter (± errore standard). Supera 0.937?
3. Il confronto appaiato: su quante immagini migliora, su quante peggiora?
4. Il guadagno sul jitter è più grande di quello sui box esatti? (atteso: sì, il training usa box con jitter)
5. I 4 fallimenti della Task 1 sono risolti?
6. Efficienza: parametri addestrati, secondi per epoca, picco di memoria, tempo della cache.